# Text Splitters in LangChain

# 1. Introduction

A **Text Splitter** in LangChain is a component used to divide large documents into smaller pieces called **chunks**.

Text splitting is an important step in many LLM applications, especially:

- RAG (Retrieval-Augmented Generation)
- Semantic search
- Question answering
- Document summarization
- Knowledge-base systems
- Vector database applications

The basic workflow is:

    Document
       ↓
    Text Splitter
       ↓
    Smaller Chunks
       ↓
    Embeddings
       ↓
    Vector Store
       ↓
    Retriever
       ↓
    LLM

---

# 2. Why Do We Need Text Splitters?

Large documents can contain thousands or millions of characters.

Sending an entire document to an LLM or embedding model can cause several problems:

- Context-window limitations
- High token usage
- Higher API cost
- Poor retrieval precision
- Slow processing
- Irrelevant information being included
- Difficult document retrieval

Instead of storing the entire document as one large piece, we divide it into smaller meaningful chunks.

Example:

    Large Document
          ↓
    ┌─────┬─────┬─────┬─────┐
    ↓     ↓     ↓     ↓     ↓
    C1    C2    C3    C4    C5

Each chunk can then be embedded and retrieved independently.

---

# 3. What is a Chunk?

A **chunk** is a smaller piece of a larger document.

For example:

    Original Document

    "LangChain is a framework for developing applications
    powered by language models. It provides components for
    prompts, models, retrieval, agents, memory, and more."

After splitting:

    Chunk 1:
    "LangChain is a framework for developing applications"

    Chunk 2:
    "powered by language models. It provides components"

    Chunk 3:
    "for prompts, models, retrieval, agents, memory, and more."

The exact chunks depend on the splitter configuration.

---

# 4. Text Splitting in RAG

Text splitting is especially important in RAG.

A typical RAG ingestion pipeline is:

    PDF / Website / TXT / DOCX
                ↓
          Document Loader
                ↓
             Documents
                ↓
          Text Splitter
                ↓
              Chunks
                ↓
           Embedding Model
                ↓
           Vector Database
                ↓
             Retriever
                ↓
          Relevant Chunks
                ↓
              Prompt
                ↓
               LLM
                ↓
             Answer

Text splitting happens during the **ingestion/indexing stage**.

---

# 5. Why Not Embed the Whole Document?

Suppose a 500-page book is stored as one vector.

A user asks:

    "What is recursion?"

Searching one vector representing the entire book is not useful.

Instead:

    500-page book
          ↓
      Split into
      thousands of chunks
          ↓
      Embed each chunk
          ↓
      Store vectors

Now retrieval can find the specific chunks discussing recursion.

---

# 6. Main Goals of Text Splitting

A good text splitter should ideally:

1. Produce manageable chunks.
2. Preserve meaningful context.
3. Avoid breaking sentences unnecessarily.
4. Preserve document structure when possible.
5. Create chunks suitable for embedding.
6. Produce useful retrieval units.

The goal is **not simply to cut text into equal pieces**.

The goal is:

> Create chunks that are small enough for efficient retrieval but large enough to preserve the context needed to answer questions.

---

# 7. Important Text Splitting Parameters

The most important parameters are:

- `chunk_size`
- `chunk_overlap`
- `length_function`
- `separators`
- `is_separator_regex`

Understanding these is essential.

---

# 8. `chunk_size`

`chunk_size` controls the maximum size of each chunk according to the splitter's length function.

Example:

    chunk_size = 1000

This means the splitter attempts to create chunks around that size.

Important:

> `chunk_size` does not necessarily mean 1000 tokens.

For many character-based splitters, it means approximately **1000 characters**.

---

# 9. Characters vs Tokens

This is a very important distinction.

If:

    chunk_size = 1000

that does not automatically mean:

    1000 tokens

It may mean:

    approximately 1000 characters

depending on the splitter.

Tokens and characters are different.

For example:

    "LangChain is powerful."

could be represented as:

    Characters → one length
    Tokens     → another length

Therefore, always understand what the splitter's `length_function` measures.

---

# 10. `chunk_overlap`

`chunk_overlap` specifies how much content should overlap between consecutive chunks.

Example:

    chunk_size = 1000
    chunk_overlap = 200

Conceptually:

    Chunk 1
    ┌──────────────────────────────┐
    │ A B C D E F G H I J          │
    └──────────────────────────────┘
                  │
                  │ 200 overlap
                  ↓
             Chunk 2
             ┌──────────────────────────────┐
             │ I J K L M N O P Q R          │
             └──────────────────────────────┘

The overlapping region helps preserve context across chunk boundaries.

---

# 11. Why Do We Need Chunk Overlap?

Suppose a sentence gets split between two chunks.

Chunk 1:

    "The heart contains four"

Chunk 2:

    "four chambers: two atria and two ventricles."

Without overlap, important context may be lost.

With overlap:

    Chunk 1:
    "The heart contains four chambers:"

    Chunk 2:
    "contains four chambers: two atria and two ventricles."

The second chunk retains some context from the previous chunk.

---

# 12. Chunk Size vs Chunk Overlap

### `chunk_size`

Controls:

> How large each chunk should be.

### `chunk_overlap`

Controls:

> How much content from the previous chunk should be repeated in the next chunk.

Example:

    chunk_size = 1000
    chunk_overlap = 200

Think:

    1000 → chunk size
     200 → shared context

---

# 13. Trade-off of Chunk Size

### Very small chunks

Advantages:

- Precise retrieval
- Less irrelevant information
- Smaller embeddings

Disadvantages:

- Context may be lost
- Important information can be separated
- More chunks are created

### Very large chunks

Advantages:

- More context
- Better preservation of relationships

Disadvantages:

- Less precise retrieval
- More irrelevant information
- Larger embedding inputs
- Potential context-window problems

Therefore:

> There is no universal perfect chunk size.

---

# 14. Trade-off of Chunk Overlap

### Too little overlap

Potential problem:

    Important context
          ↓
    Lost at boundary

### Too much overlap

Potential problems:

- Duplicate information
- More chunks
- Larger vector database
- Increased embedding cost
- Redundant retrieval

Therefore, overlap should be large enough to preserve context but not unnecessarily large.

---

# 15. Important Text Splitters in LangChain

Commonly used text splitters include:

1. `CharacterTextSplitter`
2. `RecursiveCharacterTextSplitter`
3. `TokenTextSplitter`
4. `SentenceTransformersTokenTextSplitter`
5. `MarkdownHeaderTextSplitter`
6. `HTMLHeaderTextSplitter`
7. `RecursiveJsonSplitter`
8. Language-specific splitters

The appropriate splitter depends on the structure of your data.

---

# 16. CharacterTextSplitter

`CharacterTextSplitter` splits text using a specified separator.

Example:

    from langchain_text_splitters import CharacterTextSplitter

    splitter = CharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separator="\n"
    )

    chunks = splitter.split_documents(documents)

The splitter tries to split based on the specified separator.

---

# 17. CharacterTextSplitter Concept

Suppose the input is:

    Paragraph 1

    Paragraph 2

    Paragraph 3

The separator is:

    "\n"

The splitter can use the newline as a boundary.

Conceptually:

    Document
       ↓
    Find separator
       ↓
    Create chunks
       ↓
    Return chunks

---

# 18. CharacterTextSplitter Separators

You can specify a separator.

Examples:

    separator="\n"

or:

    separator="\n\n"

or:

    separator=" "

or another appropriate delimiter.

For example:

    splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=1000,
        chunk_overlap=200
    )

This attempts to split around paragraph boundaries.

---

# 19. Limitation of CharacterTextSplitter

A simple character-based splitter does not necessarily understand the semantic structure of language.

For example, it may not understand:

- Sentences
- Paragraph meaning
- Headings
- Code structure
- HTML structure
- Markdown structure

This is why `RecursiveCharacterTextSplitter` is often a better general-purpose choice.

---

# 20. RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter` is one of the most commonly used general-purpose splitters in LangChain.

It attempts to preserve larger structural units before falling back to smaller ones.

Typical separators are conceptually:

    ┌─────────────────┐
    │ "\n\n"          │  Paragraph
    ├─────────────────┤
    │ "\n"            │  Line
    ├─────────────────┤
    │ " "             │  Word
    ├─────────────────┤
    │ ""              │  Character
    └─────────────────┘

It recursively tries these separators.

---

# 21. Why is it called "Recursive"?

Because the splitter recursively attempts to divide text using increasingly smaller separators.

Conceptually:

    Large Text
       ↓
    Try paragraph separator
       ↓
    Still too large?
       ↓
    Try line separator
       ↓
    Still too large?
       ↓
    Try word separator
       ↓
    Still too large?
       ↓
    Try character separator

This gives it more flexibility than a simple character splitter.

---

# 22. RecursiveCharacterTextSplitter Example

    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

The default separators are designed to preserve natural text boundaries where possible.

---

# 23. Why RecursiveCharacterTextSplitter is Popular

It is a good general-purpose choice because it tries to preserve:

    Paragraph
       ↓
    Line
       ↓
    Word
       ↓
    Character

rather than immediately cutting text at arbitrary character positions.

It is commonly useful for:

- PDFs
- TXT files
- General documents
- RAG applications
- Documentation
- Articles

---

# 24. CharacterTextSplitter vs RecursiveCharacterTextSplitter

| Feature | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|---|---|---|
| Basic splitting | Yes | Yes |
| Separator based | Yes | Yes |
| Multiple separators | Limited | Yes |
| Recursive splitting | No | Yes |
| General-purpose use | Good | Excellent |
| Structure preservation | Basic | Better |
| Common RAG choice | Sometimes | Very common |

### Easy rule

For general-purpose RAG:

    RecursiveCharacterTextSplitter

is often a strong starting point.

---

# 25. TokenTextSplitter

`TokenTextSplitter` splits text based on tokens rather than characters.

This can be useful when you care specifically about token limits.

Conceptually:

    Text
      ↓
    Tokenizer
      ↓
    Tokens
      ↓
    Token-based chunks

Example concept:

    from langchain_text_splitters import TokenTextSplitter

    splitter = TokenTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)

Here, the size is based on tokens according to the tokenizer used by the splitter.

---

# 26. Why Use Token-Based Splitting?

LLMs operate around token-based context limits.

Therefore, token-based splitting can be useful when:

- Context window is important
- Exact token limits matter
- Embedding/model token limits need to be respected
- You want predictable token-sized chunks

However, token splitting may break text at less natural linguistic boundaries than structure-aware splitting.

---

# 27. Character-Based vs Token-Based Splitting

### Character-based

    Text
      ↓
    Characters
      ↓
    Chunks

### Token-based

    Text
      ↓
    Tokens
      ↓
    Chunks

### Character-based advantage

Better control over textual structure when using separators.

### Token-based advantage

Better control over model/token limits.

---

# 28. MarkdownHeaderTextSplitter

Markdown documents contain structure through headers.

Example:

    # LangChain

    ## Runnables

    ## Document Loaders

    ## Text Splitters

A Markdown-aware splitter can use headers to preserve this structure.

Conceptually:

    Markdown
       ↓
    Headers
       ↓
    Structured Documents
       ↓
    Chunks

---

# 29. Why Markdown Structure Matters

Suppose a chunk contains:

    "It provides a standard interface."

Without its heading, you may not know what "it" refers to.

With metadata:

    Header:
    Runnables

the context becomes clearer.

Therefore, Markdown-aware splitting can preserve useful structural information.

---

# 30. MarkdownHeaderTextSplitter Example

Conceptually:

    from langchain_text_splitters import MarkdownHeaderTextSplitter

    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )

    chunks = splitter.split_text(markdown_text)

The resulting documents can contain header information in metadata.

---

# 31. HTMLHeaderTextSplitter

HTML documents also contain structural elements.

For example:

    <h1>LangChain</h1>

    <h2>Runnables</h2>

    <h2>Retrievers</h2>

A header-aware HTML splitter can use those structural elements when splitting.

Conceptually:

    HTML
      ↓
    Headers
      ↓
    Sections
      ↓
    Documents

This can be useful for:

- Websites
- Documentation
- Knowledge bases

---

# 32. RecursiveJsonSplitter

JSON has a hierarchical structure.

Example:

    {
        "user": {
            "name": "John",
            "address": {
                "city": "Mumbai"
            }
        }
    }

A JSON-specific splitter can recursively break large JSON structures while trying to preserve their structure.

This is useful when working with large JSON data.

---

# 33. Language-Specific Splitters

Code should not always be split like normal prose.

For programming languages, structure matters.

Examples of language-aware splitting can include:

- Python
- JavaScript
- TypeScript
- Java
- C
- C++
- Go
- Rust
- Markdown
- HTML

A language-aware splitter can prioritize code structures such as:

    Class
      ↓
    Function
      ↓
    Method
      ↓
    Statement

rather than arbitrarily splitting at random characters.

---

# 34. Why Code Needs Special Splitting

Consider:

    class Calculator:

        def add(self, a, b):
            return a + b

        def subtract(self, a, b):
            return a - b

A random character split could produce invalid fragments.

A code-aware splitter attempts to preserve meaningful structures such as:

    Class
       ↓
    Methods
       ↓
    Code Blocks

This can improve code retrieval.

---

# 35. `split_text()`

`split_text()` is used when you have a raw string and want chunks.

Example:

    text = "LangChain is a framework..."

    chunks = splitter.split_text(text)

The result is generally a list of strings.

Conceptually:

    String
      ↓
    split_text()
      ↓
    [
        "chunk 1",
        "chunk 2",
        "chunk 3"
    ]

---

# 36. `split_documents()`

`split_documents()` is used when you already have LangChain `Document` objects.

Example:

    chunks = splitter.split_documents(documents)

This is commonly used after a Document Loader.

Pipeline:

    Loader
      ↓
    Documents
      ↓
    split_documents()
      ↓
    Chunks

---

# 37. `create_documents()`

Some text splitters can also create `Document` objects from raw texts.

Conceptually:

    texts
      ↓
    create_documents()
      ↓
    Documents

This is useful when you have strings rather than existing LangChain Documents.

---

# 38. Difference Between `split_text()` and `split_documents()`

| Method | Input | Output |
|---|---|---|
| `split_text()` | String | List of text chunks |
| `split_documents()` | Documents | List of Documents |
| `create_documents()` | List of strings | List of Documents |

### Easy memory trick

    text → split_text()

    documents → split_documents()

    strings → create_documents()

---

# 39. Metadata Preservation

One major advantage of splitting Documents rather than raw strings is metadata preservation.

Suppose the original Document contains:

    metadata = {
        "source": "python.pdf",
        "page": 10
    }

After splitting:

    Chunk 1
    metadata:
        source = python.pdf
        page = 10

    Chunk 2
    metadata:
        source = python.pdf
        page = 10

This allows retrieved chunks to retain their source information.

---

# 40. Metadata After Splitting

The complete flow can be:

    PDF
      ↓
    PDF Loader
      ↓
    Document
      │
      ├── page_content
      └── metadata
              ↓
        Text Splitter
              ↓
       ┌──────┴──────┐
       ↓             ↓
    Chunk 1        Chunk 2
       ↓             ↓
    Metadata       Metadata

This is extremely important for RAG citations and source tracking.

---

# 41. Chunking Strategy

There is no single perfect chunking strategy.

The best strategy depends on:

- Data type
- Document structure
- Query type
- Embedding model
- LLM context window
- Retrieval method
- Application requirements

For example:

### Technical documentation

Prefer:

    Markdown/Header-aware
          +
    Recursive splitting

### PDFs

Often start with:

    RecursiveCharacterTextSplitter

and inspect extraction quality.

### Code

Prefer:

    Language-aware splitter

### Large structured JSON

Consider:

    RecursiveJsonSplitter

---

# 42. Semantic Chunking

Traditional chunking uses structural rules such as:

    characters
    paragraphs
    sentences
    tokens
    headers

**Semantic chunking** attempts to split text based more directly on changes in meaning.

Conceptually:

    Text
      ↓
    Embeddings / semantic similarity
      ↓
    Detect topic changes
      ↓
    Semantic Chunks

Example:

    Topic A
    Topic A
    Topic A
         ↓
    Topic changes
         ↓
    Topic B
    Topic B

Semantic chunking can produce chunks that better align with conceptual boundaries, although it may require additional computation.

---

# 43. Fixed-Size Chunking

The simplest approach is fixed-size chunking.

Example:

    chunk_size = 500

Conceptually:

    0 ───────── 500
    500 ─────── 1000
    1000 ────── 1500

Advantages:

- Simple
- Predictable
- Easy to implement

Disadvantages:

- Can split sentences
- Can break semantic units
- May lose context

---

# 44. Structure-Based Chunking

Structure-based chunking uses document boundaries.

For example:

    Document
      ↓
    Heading
      ↓
    Section
      ↓
    Paragraph
      ↓
    Sentence

This generally preserves more semantic structure than arbitrary fixed-size splitting.

---

# 45. Hierarchical Chunking

A document can be represented at multiple levels.

Example:

    Document
       ↓
    Chapter
       ↓
    Section
       ↓
    Paragraph
       ↓
    Sentence

A hierarchical retrieval system can use these levels to provide broader context around a relevant chunk.

This is useful for large technical or knowledge-heavy documents.

---

# 46. Parent-Child Chunking

Another strategy is to use:

    Parent Document
          ↓
    Child Chunks

The child chunks can be used for precise retrieval while the parent provides additional context.

Conceptually:

    Parent
    ├── Child 1
    ├── Child 2
    ├── Child 3
    └── Child 4

Retrieval:

    Query
      ↓
    Child Chunk
      ↓
    Find Parent
      ↓
    Return broader context

This can help balance retrieval precision and context.

---

# 47. Chunk Size Selection

There is no universal value such as:

    chunk_size = 1000

that works for every application.

You should consider:

### Short FAQ documents

Smaller chunks may work well.

### Technical documentation

Moderate chunks with structural boundaries may work better.

### Long reports

Larger context-aware chunks may be useful.

### Code

Structure-aware chunks are generally preferable.

---

# 48. Chunk Overlap Selection

A common starting point might be:

    chunk_size = 1000
    chunk_overlap = 100 or 200

But these are only starting points.

The ideal overlap depends on:

- Sentence length
- Document structure
- Query type
- Retrieval behavior
- Chunk size

Do not blindly use a fixed overlap for every project.

---

# 49. Common Text Splitter Pipeline

A common implementation is:

    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_documents(documents)

Now:

    documents
       ↓
    splitter
       ↓
    chunks

---

# 50. Inspecting Chunks

Always inspect the chunks after splitting.

Example:

    for i, chunk in enumerate(chunks[:5]):
        print(f"Chunk {i}")
        print(chunk.page_content)
        print(chunk.metadata)
        print("-" * 50)

Check:

- Chunk length
- Sentence boundaries
- Context
- Metadata
- Duplicate content
- Unexpected breaks

---

# 51. Measuring Chunk Size

You can inspect the length of chunks.

Example:

    for chunk in chunks:
        print(len(chunk.page_content))

This helps understand how the splitter actually behaves.

Remember:

    len(text)

usually measures Python string characters, not tokens.

---

# 52. Chunking and Embeddings

After splitting:

    Documents
       ↓
    Chunks
       ↓
    Embeddings

Each chunk is transformed into a vector.

Conceptually:

    Chunk 1 → Vector 1
    Chunk 2 → Vector 2
    Chunk 3 → Vector 3
    Chunk 4 → Vector 4

These vectors are then stored in a vector database.

---

# 53. Why Chunk Quality Affects Retrieval

Suppose a chunk contains:

    "It is used for this."

This is a poor chunk because the phrase "it" has no clear reference.

A better chunk might contain:

    "RunnableParallel executes multiple Runnable branches
    using the same input."

This chunk has enough context to be meaningful independently.

Therefore:

> Better chunks generally provide better retrieval units.

---

# 54. Chunk Quality and RAG Accuracy

The RAG pipeline can be simplified as:

    Document
       ↓
    Chunking
       ↓
    Embeddings
       ↓
    Retrieval
       ↓
    Context
       ↓
    LLM
       ↓
    Answer

If chunking is poor:

    Poor Chunks
       ↓
    Poor Retrieval
       ↓
    Poor Context
       ↓
    Poor Answer

Therefore, improving RAG quality is not only about choosing a better LLM.

Chunking can have a major impact.

---

# 55. Common Chunking Problems

## Problem 1: Chunks are too small

Symptoms:

- Missing context
- Fragmented sentences
- Retrieval returns incomplete information

Solution:

- Increase chunk size
- Increase overlap
- Use structure-aware splitting

---

## Problem 2: Chunks are too large

Symptoms:

- Too much irrelevant context
- Poor retrieval precision
- Larger embedding inputs

Solution:

- Reduce chunk size
- Use hierarchical/semantic strategies
- Preserve meaningful boundaries

---

## Problem 3: Excessive overlap

Symptoms:

- Many duplicate chunks
- Larger vector database
- Higher embedding cost

Solution:

- Reduce overlap

---

## Problem 4: Arbitrary sentence breaks

Symptoms:

    "LangChain is a framework for"

    "building LLM applications..."

Solution:

Use a recursive or structure-aware splitter.

---

## Problem 5: Lost metadata

Symptoms:

- Cannot identify source
- Difficult to provide citations
- Difficult to debug retrieval

Solution:

Use `split_documents()` when working with Documents and preserve metadata.

---

# 56. Document Loader + Text Splitter

These two components work together.

### Document Loader

Answers:

> Where does the data come from?

Example:

    PDF
      ↓
    PyPDFLoader

### Text Splitter

Answers:

> How should the loaded data be divided?

Example:

    Documents
       ↓
    RecursiveCharacterTextSplitter

Complete flow:

    PDF
      ↓
    PDF Loader
      ↓
    Documents
      ↓
    Text Splitter
      ↓
    Chunks

---

# 57. Text Splitter + Embedding Model

After splitting:

    Chunks
      ↓
    Embedding Model
      ↓
    Vectors

The embedding model converts each chunk into a numerical representation.

Example:

    "LangChain is a framework..."
              ↓
        Embedding Model
              ↓
    [0.021, -0.341, 0.827, ...]

The vector can then be stored in a vector database.

---

# 58. Complete RAG Ingestion Pipeline

    ┌────────────────────┐
    │     Data Source    │
    │ PDF/TXT/HTML/etc. │
    └──────────┬─────────┘
               ↓
    ┌────────────────────┐
    │  Document Loader   │
    └──────────┬─────────┘
               ↓
    ┌────────────────────┐
    │     Documents      │
    └──────────┬─────────┘
               ↓
    ┌────────────────────┐
    │   Text Splitter    │
    └──────────┬─────────┘
               ↓
    ┌────────────────────┐
    │       Chunks       │
    └──────────┬─────────┘
               ↓
    ┌────────────────────┐
    │    Embeddings      │
    └──────────┬─────────┘
               ↓
    ┌────────────────────┐
    │    Vector Store    │
    └────────────────────┘

---

# 59. Example: PDF → Chunks

    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    # Load PDF
    loader = PyPDFLoader("document.pdf")

    documents = loader.load()

    # Create splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    # Split documents
    chunks = splitter.split_documents(documents)

    print("Documents:", len(documents))
    print("Chunks:", len(chunks))

---

# 60. Example: Text → Chunks

    from langchain_text_splitters import RecursiveCharacterTextSplitter

    text = """
    LangChain is a framework for developing applications
    powered by language models.

    It provides components for prompts, models, retrieval,
    agents, and other LLM application workflows.
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20
    )

    chunks = splitter.split_text(text)

    for chunk in chunks:
        print(chunk)
        print("---")

---

# 61. Example: Markdown Chunking

    from langchain_text_splitters import MarkdownHeaderTextSplitter

    markdown_text = """
    # LangChain

    LangChain is a framework.

    ## Runnables

    Runnables provide a standard interface.

    ## Retrievers

    Retrievers fetch relevant documents.
    """

    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2")
    ]

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )

    documents = splitter.split_text(markdown_text)

The resulting Documents can preserve heading information in metadata.

---

# 62. Example: Token-Based Splitting

    from langchain_text_splitters import TokenTextSplitter

    splitter = TokenTextSplitter(
        chunk_size=500,
        chunk_overlap=50
    )

    chunks = splitter.split_documents(documents)

Here the chunk size is token-oriented rather than character-oriented.

---

# 63. Recommended Starting Strategy

For a general RAG project, a reasonable starting point is:

    RecursiveCharacterTextSplitter

with something like:

    chunk_size = 500–1500
    chunk_overlap = 50–200

These are starting points, not universal rules.

Then evaluate:

- Retrieval precision
- Retrieval recall
- Answer quality
- Context completeness
- Token usage
- Latency
- Cost

---

# 64. Advanced Chunking Strategy

A more mature RAG system can use:

    Document Loader
          ↓
    Structure Detection
          ↓
    Header / Section Splitting
          ↓
    Recursive Splitting
          ↓
    Semantic Validation
          ↓
    Chunk Metadata
          ↓
    Embeddings
          ↓
    Vector Store

This can produce better retrieval than blindly splitting every document using one fixed size.

---

# 65. Chunking Evaluation

Do not evaluate chunking only by looking at chunk size.

Evaluate:

### Retrieval

Does the correct chunk get retrieved?

### Context

Does the chunk contain enough information?

### Precision

Does the retrieved chunk contain unnecessary information?

### Recall

Can the system find the information needed to answer?

### Answer quality

Does the LLM generate the correct answer from the retrieved context?

### Cost

How many chunks and embeddings are being generated?

---

# 66. Important Principle

> **Chunking is an information-retrieval problem, not merely a string-splitting problem.**

The purpose of chunking is to create useful retrieval units.

Therefore, the "best" chunk is not necessarily:

    exactly 1000 characters

The best chunk is one that:

- Represents a coherent piece of information
- Contains sufficient context
- Can be retrieved accurately
- Does not contain excessive unrelated information

---

# 67. Text Splitter Decision Tree

Use this simple decision process:

    What type of data?
          ↓
    ┌─────┼──────────────┬──────────────┐
    ↓     ↓              ↓              ↓
   Text  Markdown       HTML           Code
    ↓     ↓              ↓              ↓
 Recursive Header      Header        Language
 Character Splitter    Splitter       Splitter

For token-sensitive applications:

    TokenTextSplitter

For large JSON:

    RecursiveJsonSplitter

For semantic boundaries:

    Semantic Chunking

---

# 68. Text Splitters Quick Comparison

| Splitter | Best Use |
|---|---|
| `CharacterTextSplitter` | Simple delimiter-based splitting |
| `RecursiveCharacterTextSplitter` | General-purpose text and RAG |
| `TokenTextSplitter` | Token-aware chunking |
| `MarkdownHeaderTextSplitter` | Markdown documentation |
| `HTMLHeaderTextSplitter` | Structured HTML |
| `RecursiveJsonSplitter` | Large JSON |
| Language-specific splitters | Source code |
| Semantic chunking | Meaning-based boundaries |

---

# 69. Most Important Interview Questions

## Q1. What is a Text Splitter?

A Text Splitter divides large documents into smaller chunks that can be processed, embedded, stored, and retrieved efficiently.

---

## Q2. Why is text splitting important in RAG?

Because large documents are difficult to embed and retrieve effectively as a single unit. Splitting creates smaller retrieval units that can provide more relevant context to the LLM.

---

## Q3. What is `chunk_size`?

`chunk_size` controls the target/maximum size of chunks according to the splitter's length function.

---

## Q4. What is `chunk_overlap`?

`chunk_overlap` specifies how much content from one chunk is repeated in the following chunk to help preserve context across boundaries.

---

## Q5. What is `RecursiveCharacterTextSplitter`?

It is a general-purpose splitter that recursively attempts to split text using increasingly smaller separators, helping preserve natural text boundaries.

---

## Q6. Why is RecursiveCharacterTextSplitter commonly used?

Because it attempts to preserve larger units such as paragraphs and lines before falling back to smaller units such as words and characters.

---

## Q7. What is the difference between `split_text()` and `split_documents()`?

`split_text()` takes raw text and returns text chunks.

`split_documents()` takes LangChain Documents and returns split Documents while allowing document metadata to be preserved.

---

## Q8. What is TokenTextSplitter?

A splitter that divides text based on tokens rather than characters.

---

## Q9. Why is chunk overlap useful?

It helps preserve context when important information lies across chunk boundaries.

---

## Q10. Is a larger chunk size always better?

No.

Larger chunks provide more context but can reduce retrieval precision and introduce irrelevant information.

---

## Q11. Is a smaller chunk size always better?

No.

Smaller chunks can improve precision but may lose important context.

---

## Q12. Is there a universal best chunk size?

No.

Chunk size depends on:

- Data
- Query type
- Embedding model
- LLM
- Retrieval strategy
- Application requirements

---

# 70. Common Mistakes

## Mistake 1: Treating `chunk_size` as tokens

If a character-based splitter is being used:

    chunk_size = 1000

does not necessarily mean:

    1000 tokens

Always check the length function.

---

## Mistake 2: Using the same splitter for every data type

PDF, Markdown, HTML, JSON, and source code have different structures.

Use structure-aware splitters when appropriate.

---

## Mistake 3: Using huge overlap

Too much overlap creates unnecessary duplication.

---

## Mistake 4: Using tiny chunks

Tiny chunks may lose the context required to answer questions.

---

## Mistake 5: Not inspecting chunks

Always inspect actual output.

Never assume the splitter created good chunks simply because no error occurred.

---

## Mistake 6: Ignoring metadata

Metadata is valuable for:

- Source citations
- Filtering
- Debugging
- Document tracking

---

# 71. Best Practices

### 1. Start simple

For general RAG, begin with:

    RecursiveCharacterTextSplitter

### 2. Inspect your chunks

Look at real output rather than relying only on configuration.

### 3. Preserve metadata

Keep source and page information whenever possible.

### 4. Tune experimentally

Try different:

    chunk_size
    chunk_overlap

and evaluate retrieval quality.

### 5. Respect document structure

Use:

- Markdown splitters for Markdown
- HTML splitters for HTML
- Code splitters for source code
- JSON splitters for structured JSON

### 6. Consider token limits

Make sure chunks are compatible with your embedding and model limits.

### 7. Evaluate retrieval

The ultimate test is:

> Does the retriever return the information needed to answer the question?

---

# 72. Complete Mental Model

Remember Text Splitters using:

    LARGE DOCUMENT
          ↓
    TEXT SPLITTER
          ↓
    ┌─────┬─────┬─────┬─────┐
    ↓     ↓     ↓     ↓     ↓
   C1    C2    C3    C4    C5
    ↓     ↓     ↓     ↓     ↓
 Embedding Embedding Embedding ...
    ↓
 Vector Store
    ↓
 Retriever
    ↓
 Relevant Chunks
    ↓
 LLM
    ↓
 Answer

---

# 73. Final Summary

A **Text Splitter** divides large documents into smaller chunks suitable for downstream processing.

The most important concepts are:

    Document
       ↓
    Text Splitter
       ↓
    Chunks

Important parameters:

    chunk_size
    chunk_overlap
    length_function
    separators

Important splitters:

    CharacterTextSplitter
    RecursiveCharacterTextSplitter
    TokenTextSplitter
    MarkdownHeaderTextSplitter
    HTMLHeaderTextSplitter
    RecursiveJsonSplitter
    Language-specific splitters

Important methods:

    split_text()
    split_documents()
    create_documents()

For general-purpose RAG, a strong starting point is:

    RecursiveCharacterTextSplitter

But the final chunking strategy should be selected and evaluated based on the actual data and retrieval requirements.

---

# 74. One-Line Definition

> **A Text Splitter in LangChain is a component that divides large documents into smaller, meaningful chunks so they can be efficiently embedded, stored, retrieved, and supplied as relevant context to an LLM.**

---

# 75. Ultimate RAG Formula

    LOAD
      ↓
    SPLIT
      ↓
    EMBED
      ↓
    STORE
      ↓
    RETRIEVE
      ↓
    AUGMENT
      ↓
    GENERATE

In this pipeline:

    Document Loader → LOAD

    Text Splitter → SPLIT

    Embedding Model → EMBED

    Vector Store → STORE

    Retriever → RETRIEVE

    Prompt + Retrieved Context → AUGMENT

    LLM → GENERATE

Therefore:

> **Document Loaders bring knowledge into the system, Text Splitters turn that knowledge into retrieval-friendly chunks, Embeddings turn chunks into vectors, and Retrievers find the most relevant chunks for the LLM.**